# AI-Driven Structural (AST) Trojan Insertion

**CSAW 2026 AI Hardware Attack Challenge — The Bitstream Bandits**

Instead of hand-wrapping the recovered core in a new top module, the AI emits a
single **net-tap module** and a script splices it **structurally** into the
recovered netlist: the target net's driver is rerouted through an inserted
instance, validated with **Yosys**. The splice tooling (`tb/trojan_pipeline/`) is
content-neutral — it applies whatever tap module the AI returns; the
trigger/payload behaviour lives only in the AI-generated module.

Pipeline: **AI emits tap module → Yosys analyses the net → script splices →
Yosys re-validates → Icarus compiles**. The mechanism is checked offline by
`tb/trojan_pipeline/selftest.py` (a transparent pass-through tap that must
preserve encrypt/decrypt).

## 1. Setup

In [ ]:
from pathlib import Path
import os, re, json, sys, subprocess
from datetime import datetime, timezone


def find_repo_root(markers, g):
    starts = []
    for var in ("__vsc_ipynb_file__", "__file__", "__session__"):
        val = g.get(var)
        if val:
            try:
                starts.append(Path(val).resolve().parent)
            except Exception:
                pass
    starts.append(Path.cwd().resolve())
    for start in starts:
        for d in (start, *start.parents):
            for sub in ("", "aha2026-dp1"):
                cand = d / sub if sub else d
                if all((cand / m).exists() for m in markers):
                    return cand
    raise FileNotFoundError(f"Open this notebook inside the aha2026-dp1 repository (cwd={Path.cwd()})")


ROOT = find_repo_root(("recovery/recovered_sim.v", "tb/trojan_pipeline"), globals())
if str(ROOT / "tb") not in sys.path:
    sys.path.insert(0, str(ROOT / "tb"))

from trojan_pipeline import (analyze_net, splice_net_tap, validate, build_net_tap_prompt,
                             extract_module, find_tool, find_cells, NET_TAP_INTERFACE, SYSTEM_MESSAGE)
from portkey_ai import Portkey

MODEL = os.getenv("PORTKEY_MODEL", "@vertexai/gemini-3.1-pro-preview")
_keyfile = ROOT / ".env.portkey.json"
API_KEY = os.environ.get("PORTKEY_API_KEY") or (
    json.loads(_keyfile.read_text())["PORTKEY_API_KEY"] if _keyfile.is_file() else "")
client = Portkey(api_key=API_KEY,
                 base_url=os.getenv("PORTKEY_BASE_URL", "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"),
                 max_retries=0)

CORE = ROOT / "recovery/recovered_sim.v"
CELLS = find_cells()
OUT = ROOT / "rtl/trojans/ast";            OUT.mkdir(parents=True, exist_ok=True)
LOG = ROOT / "ai_logs/trojan_pipeline";    LOG.mkdir(parents=True, exist_ok=True)
BUILD = ROOT / "build/trojan_pipeline";    BUILD.mkdir(parents=True, exist_ok=True)

# the net to tap and the observable SPI signals wired into the tap
TARGET_NET = "pin_31"   # MISO
OBSERVE = {"sck": "pin_36", "mosi": "pin_32", "cs_n": "pin_28", "rst_n": "pin_34"}

print("Repository :", ROOT)
print("Model      :", MODEL)
print("iCE40 cells:", CELLS)
print("Target net :", TARGET_NET, "(MISO)")
print("Key loaded :", bool(API_KEY))

## 2. Trojan behaviour specification

The behaviour the AI should implement, described at the spec level (exactly as
`notebooks/trojans/*.ipynb` supply a `SPEC` string). **Edit this for your
variant** — the tooling and prompt are behaviour-agnostic.

In [ ]:
SPEC = (
    "Trigger: a complete 32-bit SPI transaction whose shifted-in word equals "
    "0xA6F03C59. Payload: for the single next transaction only, drive `patched` "
    "as the bitwise inverse of `orig`; afterwards return to transparent "
    "(`patched = orig`). The trigger transaction itself stays transparent."
)
print(SPEC)

## 3. AI emits the net-tap module (the patch)

The model returns one `net_tap` module with the fixed interface. `extract_module`
rejects anything that doesn't match the contract before it is used.

In [ ]:
REGENERATE = True          # False reuses an existing rtl/trojans/ast/net_tap.sv
tap_path = OUT / "net_tap.sv"

if REGENERATE or not tap_path.is_file():
    prompt = build_net_tap_prompt(SPEC)
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
    run = LOG / run_id
    run.mkdir(exist_ok=True)
    (run / "prompt.txt").write_text(f"Model: {MODEL}\n\n{SYSTEM_MESSAGE}\n\n{prompt}")
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=8000, timeout=300,
        messages=[{"role": "system", "content": SYSTEM_MESSAGE},
                  {"role": "user", "content": prompt}])
    raw = resp.choices[0].message.content or ""
    (run / "response.txt").write_text(raw)
    (run / "response_raw.json").write_text(resp.model_dump_json(indent=2))
    reason = str(resp.choices[0].finish_reason).lower()
    if reason not in ("none", "stop", "end_turn"):
        raise RuntimeError(f"Model finish reason: {reason}; see {run}")
    tap_code = extract_module(raw)      # validates the net_tap interface
    tap_path.write_text(tap_code)
    print("AI emitted net_tap ->", tap_path.relative_to(ROOT), f"({len(tap_code)} chars)")
    print("transcript ->", run.relative_to(ROOT))
else:
    print("Reusing", tap_path.relative_to(ROOT))
print("\n--- interface the AI was required to use ---")
print(NET_TAP_INTERFACE)

## 4. Structural AST insertion

Yosys analyses the target net, then the script reroutes its driver through the
AI's module (renaming the original driver to `<net>__pre` and inserting the
instance) and re-validates — a structural edit of the recovered module, not a
wrapper.

In [ ]:
facts = analyze_net(CORE, TARGET_NET)
print("Yosys analysis:", facts)

patched, report = splice_net_tap(CORE.read_text(), TARGET_NET, "net_tap", OBSERVE)
print("Structural splice:", report)

patched_path = BUILD / "recovered_tapped.v"
patched_path.write_text(patched)

ok, log = validate(patched_path, tap_path, CELLS)
print("Yosys re-validation of patched netlist:", "OK" if ok else "FAIL")
assert ok, log[-800:]
print("patched netlist ->", patched_path.relative_to(ROOT))

## 5. Compile check

The spliced netlist plus the AI tap must elaborate for iCE40.

In [ ]:
cmd = [find_tool("iverilog"), "-g2012", "-DNO_ICE40_DEFAULT_ASSIGNMENTS",
       "-s", "recovered", "-t", "null", str(patched_path), str(tap_path), CELLS]
r = subprocess.run(cmd, capture_output=True, text=True, cwd=BUILD)
print("Compilation:", "PASS" if r.returncode == 0 else "FAIL")
if r.stderr.strip():
    print(r.stderr[-1500:])
assert r.returncode == 0, "structural insertion did not compile"

## 6. Result

The Trojan is now spliced **structurally** into net `pin_31` of module
`recovered` — no top-level wrapper. The AI-emitted tap is under
`rtl/trojans/ast/`, its transcript under `ai_logs/trojan_pipeline/`, and the
patched netlist under `build/trojan_pipeline/`.

In [ ]:
print("AI-emitted tap module :", tap_path.relative_to(ROOT))
print("Patched netlist       :", patched_path.relative_to(ROOT), "(module recovered, structurally tapped)")
print("Tapped net            :", TARGET_NET)
print()
print("To simulate the exploit, drive this patched netlist + tap through")
print("notebooks/trojan_verification.ipynb (crypto_dut instantiates `recovered`).")